# Lab 3 — Notebook 01: Models and Loss Functions

**Course:** Artificial Intelligence with Python  
**Estimated duration:** 50 minutes  
**Prerequisites:** Labs 1 and 2; basic NumPy, Pandas, and Matplotlib  
**Required data:** `data/linear_rent.csv`

In Lab 2, you prepared datasets for training. Now we will investigate **what learning means** before using ready-made machine-learning algorithms.

**By the end of this notebook you should be able to:**
- Distinguish input data, targets, predictions, and model parameters.
- Use a simple linear model to predict apartment rents.
- Calculate errors, mean absolute error (MAE), and mean squared error (MSE).
- Explain why a loss function is necessary for training.
- Experiment manually with parameters to reduce loss.

**How to work:** Run the example cells in order. Complete cells labelled `# TODO` yourself. The examples run without completing the exercises; use Markdown answers to explain what you observe.

> This dataset is **synthetic** and deliberately simple. It is separate from the corrupted apartment dataset in Lab 2 so that we can focus on how model parameters affect predictions. We will reuse it in the gradient-descent notebook.

## 1. Load the data and identify the learning task (8 min)

We have recorded apartment floor areas and monthly rents. We want a model that predicts the **rent** using only the **area**.

- **Input / feature** `x`: `area_m2`
- **Observed target** `y`: `monthly_rent_eur`
- **Model output** `ŷ`: estimated monthly rent

Unlike classification, this is a **regression problem** because the predicted value is continuous.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

path = Path("data/linear_rent.csv")
if not path.exists():
    raise FileNotFoundError(
        f"Cannot find {path}. Start JupyterLab in the lab03 folder "
        "or open this notebook with lab03 as the working directory."
    )

df = pd.read_csv(path)
print("Rows and columns:", df.shape)
print(df.head())
print("Missing values:\n", df.isna().sum())

In [ ]:
x = df["area_m2"].to_numpy(dtype=float)
y = df["monthly_rent_eur"].to_numpy(dtype=float)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x, y, alpha=0.75)
ax.set(xlabel="Floor area (m²)", ylabel="Monthly rent (€)",
       title="Synthetic apartment rents")
ax.grid(alpha=0.2)
plt.show()

### Exercise 1 — Read the data (3 min)

1. Find the smallest and largest apartment areas.
2. Calculate the average monthly rent.
3. Based on the scatter plot, does rent generally increase or decrease with area?

Use NumPy or Pandas. Write your answer to question 3 in the Markdown cell.

In [ ]:
# TODO: print the minimum and maximum areas
# TODO: print the average monthly rent

**Observation:** TODO — Describe the relationship between area and rent.

## 2. Build a model and make predictions (12 min)

Our first model is a straight line:

\[
\hat{y} = wx + b
\]

Here:

- `w` is the **weight** (slope): predicted change in monthly rent for one additional square metre.
- `b` is the **bias** (intercept): predicted rent when the area is zero.
- The pair `(w, b)` contains the model's **parameters**.

We choose the parameters *ourselves* for now. We are **not training** yet.

> A model can make a prediction with any chosen parameters. **Learning** means finding parameters that produce better predictions according to a loss function.

In [ ]:
def predict_rent(area, weight, bias):
    """Return predicted rents for a scalar or NumPy array of areas."""
    return weight * area + bias

weight = 10.0
bias = 350.0

print("Prediction for 50 m²:", predict_rent(50, weight, bias), "€")
predictions = predict_rent(x, weight, bias)
print("First five predictions:", predictions[:5])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x, y, alpha=0.65, label="Observed rents")
order = np.argsort(x)
ax.plot(x[order], predictions[order], color="tab:orange", linewidth=2,
        label=f"Model: rent = {weight:g} × area + {bias:g}")
ax.set(xlabel="Floor area (m²)", ylabel="Monthly rent (€)",
       title="Observed values vs. the model's predictions")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

### Exercise 2 — Change the parameters (4 min)

1. Try `weight = 14` and `bias = 300` **in the exercise cell below**.
2. Plot your model over the observed data (you can adapt the previous plot).
3. Estimate a reasonable value for the weight from the data. What does its unit mean?

Do **not** change the guided example above: later examples use their own explicit parameter values.

In [ ]:
# TODO: set your own weight and bias here
# student_weight = ...
# student_bias = ...
# TODO: calculate predictions for all x values
# TODO: plot the observed data and your line

**Interpretation:** TODO — Describe what changes when (a) weight increases and (b) bias increases.

## 3. Measure prediction error with loss functions (12 min)

A model's predictions rarely match all observed rents exactly. For each apartment, the **residual** (signed error) is:

\[
 e_i = y_i - \hat{y}_i
\]

We need a single number to judge how well our model fits the observations.

**Mean absolute error (MAE)** averages the *sizes* of the errors:

\[
\mathrm{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i-\hat y_i|
\]

**Mean squared error (MSE)** averages *squared* errors:

\[
\mathrm{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat y_i)^2
\]

Smaller values mean a better fit **on the evaluated data**. MAE is measured in euros, while MSE is measured in euros squared. Squaring gives relatively more weight to large errors.

In [ ]:
# Start with a tiny example we can check by hand.
actual = np.array([800., 1000., 1200.])
predicted = np.array([850., 950., 1100.])

residuals = actual - predicted
mae = np.mean(np.abs(residuals))
mse = np.mean(residuals ** 2)

print("Residuals (€):", residuals)
print(f"MAE: {mae:.2f} €")
print(f"MSE: {mse:.2f} €²")

In [ ]:
def mean_absolute_error(actual, predicted):
    return float(np.mean(np.abs(actual - predicted)))

def mean_squared_error(actual, predicted):
    return float(np.mean((actual - predicted) ** 2))

model_a = predict_rent(x, weight=10, bias=350)
model_b = predict_rent(x, weight=14, bias=300)

comparison = pd.DataFrame({
    "model": ["A: w=10, b=350", "B: w=14, b=300"],
    "MAE_eur": [mean_absolute_error(y, model_a), mean_absolute_error(y, model_b)],
    "MSE_eur2": [mean_squared_error(y, model_a), mean_squared_error(y, model_b)],
})
print(comparison.round(2).to_string(index=False))

### Exercise 3 — Compare the error measures (4 min)

1. Manually calculate the MAE for `actual = [800, 1000, 1200]` and `predicted = [850, 950, 1100]`.
2. Calculate the MAE and MSE for a **third model** with `weight=12` and `bias=400`.
3. Is the same model best according to both error measures?

**Question:** Why does MSE penalize one very large prediction error more heavily than MAE?

In [ ]:
# TODO: create the predictions for the third model
# TODO: print its MAE and MSE using the helper functions

**Your explanation:** TODO — Compare MAE and MSE, including their units.

## 4. Search for better parameters (12 min)

Now we will improve a model **automatically by trying candidates**. This is a simple parameter search, *not* gradient descent.

For a fair comparison, hold the bias fixed at `320` and try different weights. Our goal is to find the weight with the **lowest MSE**. This illustrates the idea behind minimizing a loss function.

In [ ]:
candidate_weights = np.arange(7.0, 20.1, 0.5)
fixed_bias = 320.0
losses = []

for candidate_weight in candidate_weights:
    y_hat = predict_rent(x, candidate_weight, fixed_bias)
    losses.append(mean_squared_error(y, y_hat))

losses = np.array(losses)
best_index = np.argmin(losses)
best_weight = candidate_weights[best_index]

print(f"Best tested weight: {best_weight:.1f} €/m²")
print(f"Lowest tested MSE: {losses[best_index]:.2f} €²")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(candidate_weights, losses, marker="o", markersize=3)
ax.axvline(best_weight, linestyle="--", color="tab:orange",
           label=f"Lowest tested loss at w={best_weight:g}")
ax.set(xlabel="Weight w (€/m²)", ylabel="MSE (€²)",
       title="How model loss changes with the weight")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x, y, alpha=0.55, label="Observed rents")
order = np.argsort(x)
for candidate_weight, style in [(9.0, ":"), (best_weight, "-"), (18.0, "--")]:
    line = predict_rent(x, candidate_weight, fixed_bias)
    ax.plot(x[order], line[order], style, linewidth=2,
            label=f"w={candidate_weight:g}, b={fixed_bias:g}")
ax.set(xlabel="Floor area (m²)", ylabel="Monthly rent (€)",
       title="Comparing three parameter choices")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

### Exercise 4 — Investigate the loss curve (4 min)

1. Try a **coarser search** using `np.arange(7, 21, 2)`.
2. Does it find the same best weight as the finer search?
3. If the bias were also allowed to change, could the MSE become even lower?

Note that evaluating many candidate weights requires many predictions. In Notebook 02, we will learn a more systematic way to update parameters.

In [ ]:
# TODO: create coarse_weights and compute its MSE values
# TODO: print the best coarse weight and MSE

**Observation:** TODO — Explain how the spacing between tested weights affects the answer.

## 5. Mini-challenge and reflection (6 min)

**Mini-challenge: A rent prediction for a new apartment**

Imagine a new 75 m² apartment.

1. Predict its rent using the best weight from Section 4 and the fixed bias `320`.
2. Choose a different `(weight, bias)` pair and compare its **MSE on the provided data** with the previous model.
3. Explain which model you would select and why.
4. Identify one important limitation of this experiment: are we evaluating on data that was *not* used to choose the parameters?

Use the following cell for your experiment. There is no single required choice for question 2.

In [ ]:
# TODO: predict the rent for a 75 m² apartment
# TODO: choose a second model and calculate its dataset MSE
# TODO: compare both models

**Your reflection:**

- **My selected parameters:** TODO
- **Reason for choosing them:** TODO
- **Limitation of evaluating on the same data used to choose parameters:** TODO

### Key takeaways

- A model turns an input into a prediction using **parameters**.
- A **loss function** summarizes how wrong the predictions are.
- Learning aims to **reduce a chosen loss** by adjusting model parameters.
- We chose parameters by trying many values; this is not yet gradient descent.
- A small training loss does **not** guarantee good predictions on unseen apartments. We will revisit generalization in Notebook 03.

**Next notebook:** `02-gradient-descent.ipynb` — Let the computer update weights and bias step by step instead of testing candidate values by hand.